# 🔍 TruthLens — Deepfake Detector Training (Colab GPU)

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Fill in your Kaggle credentials in **Step 3**
3. Then `Runtime → Run all`

## ✅ Step 1 — Verify GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU available: {gpus[0].name}')
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print('❌ No GPU — go to Runtime → Change runtime type → T4 GPU and restart')

## 📦 Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

MODEL_SAVE_PATH = '/content/drive/MyDrive/TruthLens/deepfake_detector_model.keras'
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
print(f'✅ Model will be saved to: {MODEL_SAVE_PATH}')

## 📥 Step 3 — Kaggle Credentials

**Fill in your username and key below.**  
Get your key from: https://www.kaggle.com/settings → Account → API → **Create New Token**

Your username is shown on your Kaggle profile page.

In [ ]:
# ✏️  FILL THESE IN:
KAGGLE_USERNAME = 'your_kaggle_username'   # e.g. 'vighneshtule'
KAGGLE_KEY      = 'your_kaggle_api_key'    # the long key from kaggle.json

# --- do not edit below this line ---
import os, json

os.makedirs('/root/.config/kaggle', exist_ok=True)
creds = {'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump(creds, f)
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

# Also set as env vars (belt-and-suspenders)
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

print('✅ Kaggle credentials set.')

In [ ]:
# Download the deepfake dataset from Kaggle (~1-3 GB, takes 2-5 min)
!pip install -q kaggle
!kaggle datasets download -d manjilkarki/deepfake-and-real-images -p /content/data --unzip
print('✅ Dataset downloaded.')

In [ ]:
# Auto-detect dataset root and verify structure
import os

DATASET_BASE = None
for root, dirs, files in os.walk('/content/data'):
    if 'Train' in dirs and 'Test' in dirs:
        DATASET_BASE = root
        break

if DATASET_BASE is None:
    # Show what was actually downloaded so we can debug
    print('❌ Could not auto-detect dataset. Showing downloaded structure:')
    for root, dirs, files in os.walk('/content/data'):
        depth = root.replace('/content/data', '').count(os.sep)
        if depth < 4:
            indent = '  ' * depth
            print(f'{indent}{os.path.basename(root)}/')
    raise RuntimeError('Fix DATASET_BASE manually — set it to the folder containing Train/Test/Validation')

print(f'✅ Dataset root: {DATASET_BASE}')
for split in ['Train', 'Validation', 'Test']:
    for cls in ['Fake', 'Real']:
        p = os.path.join(DATASET_BASE, split, cls)
        if os.path.exists(p):
            n = len(os.listdir(p))
            print(f'  {split}/{cls}: {n:,} images')

## 🧠 Step 4 — Build & Train Model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import numpy as np

tf.keras.mixed_precision.set_global_policy('mixed_float16')

IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32

def make_dataset(split_path, augment=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        split_path,
        labels='inferred',
        color_mode='rgb',
        seed=42,
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
    )
    if augment:
        aug = tf.keras.Sequential([
            layers.RandomFlip('horizontal'),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
            layers.RandomContrast(0.2),
            layers.RandomBrightness(0.2),
        ])
        ds = ds.map(lambda x, y: (aug(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


train_data = make_dataset(os.path.join(DATASET_BASE, 'Train'),      augment=True)
val_data   = make_dataset(os.path.join(DATASET_BASE, 'Validation'), augment=False)
test_data  = make_dataset(os.path.join(DATASET_BASE, 'Test'),       augment=False)
print('✅ Datasets ready.')

In [ ]:
def build_model():
    backbone = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(*IMAGE_SIZE, 3),
    )
    backbone.trainable = False

    inputs  = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x       = backbone(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)

    return tf.keras.Model(inputs, outputs)


model = build_model()
model.summary()

In [ ]:
# Phase 1 — train classifier head only (backbone frozen)
print('=== Phase 1: Classifier head (backbone frozen) ===')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

history_p1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, mode='max'),
        ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, mode='max'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7),
    ]
)
print(f'Phase 1 best val_accuracy: {max(history_p1.history["val_accuracy"]):.4f}')

In [ ]:
# Phase 2 — fine-tune top 30 EfficientNetB0 layers
print('=== Phase 2: Fine-tune top 30 layers ===')

backbone = model.layers[1]
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.0),
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

history_p2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, mode='max'),
        ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, mode='max'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-8),
    ]
)
print(f'Phase 2 best val_accuracy: {max(history_p2.history["val_accuracy"]):.4f}')

## 📊 Step 5 — Evaluate on Test Set

In [ ]:
print('=== Final Test Set Evaluation ===')
results = model.evaluate(test_data, verbose=1)
for name, val in zip(['loss', 'accuracy', 'precision', 'recall'], results):
    print(f'  {name}: {val:.4f}')

## 💾 Step 6 — Download Model to Your PC

The model is already saved to your Google Drive at `MyDrive/TruthLens/deepfake_detector_model.keras`.  
You can also download it directly:

In [ ]:
from google.colab import files
files.download(MODEL_SAVE_PATH)
print('✅ Download started!')
print('Place deepfake_detector_model.keras in your TruthLens/ folder, then run: python inference.py')